# 15 · Bits, compute and quality: where each one pays

A stream can buy quality two ways. The **traditional** way is more bits: the host gives H.264 a higher bitrate and the
client upscales with bicubic, which a GPU's texture sampler does for free. The other costs compute: an **upscaler** on
the client. This notebook puts both in one currency: **bandwidth**.

For every client and bitrate, the question is how many bits the traditional stream would need for the same quality.
That divided by the bits the client's stream really used is what the client is **worth in bandwidth**: x2 means it
looks like a plain stream at twice the bits; x1 is no better than spending the bits on the video. Where no bitrate
reaches a client's quality at all, its worth is unbounded, and the second currency says how much it adds: **quality at
the same bits**, the client's score minus what the traditional stream scores with the bits the client's stream used.
Set against what each client costs to run, the two say where compute pays most, and where the encoder alone is the
better buy.

- **Clips**: 8 validation sources of each kind of notebook 0's set (screens, photos, video), 32 frames of 1280×720
  from the centre, streamed through the host's x264 line, with the H.264 stream's real coded size.
- **Clients**: the published graphs, run as the client runs them: a tile graph on the frame cut into 320×180 tiles,
  each with its halo of real neighbours; the client's told graphs (notebooks 23 and 24) on the whole frame, told each
  frame's slice QP, loop filter and I or P as the client reads them.
- **The traditional stream** is measured further up the bitrate scale (to 3.3 bits per LR pixel), so the best client
  has a plain stream to be matched against. Where quality at a bitrate is reached by no bitrate at all (a still
  screen, which H.264 codes at the same size whatever it is given), the clip is counted apart.

Needs: `00_data_preprocess.ipynb`, the downloaded datasets, and the graphs notebooks 7, 10, 12, 16, 23 and 24 publish
in `dist/upscale/`. Writes nothing.

In [ ]:
"""Settings."""

import math
import os

import matplotlib.pyplot as plt
import numpy as np
import onnx
import torch
import torch.nn.functional as F

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import degrade, metrics, models, motion, patches, sources, training, webexport
from helpers.training import DEVICE


def first(pattern):
    """The published graph a pattern names (a notebook publishes its best under a name that says which)."""
    found = sorted(path.name for path in webexport.MODELS_DIR.glob(pattern) if "fp16" not in path.name)
    assert found, f"no graph in {webexport.MODELS_DIR} matches {pattern}: run its notebook first"
    return found[0]


CLIENTS = {   # how a client upscales: a published graph, or None for bicubic (the GPU's own sampler)
    "bicubic": None,
    "DirNet B": "upscale_dirnet_fixed_tile320x180.onnx",                 # notebook 12: one 5x5 filter
    "DirNet C": "upscale_dirnet_adaptive_tile320x180.onnx",              # notebook 12: a blend chosen per pixel
    "DirNet D": "upscale_dirnet_full_tile320x180.onnx",                  # notebook 12: C and a learned correction
    "32-16 leaky": "upscale_luma_32-16_leaky_tile320x180.onnx",          # notebook 16: the cheap tier
    "luma 24x2": "upscale_luma_tile320x180.onnx",                        # notebook 7: luma 24 + 2 residual blocks
    "look-up table": first("upscale_lut_*_tile320x180.onnx"),            # notebook 10's best table
    "told fast q24": "client/upscale_fast.onnx",                         # notebook 24: told, for 60 fps on a 1660 Ti
    "told 24x2": "client/upscale.onnx",                                  # notebook 23: what desktop-streamer runs
}
SOURCES_PER_KIND = 8
SIZE, CLIP_FRAMES, IMAGE_FPS = (1280, 720), 32, 30
BITRATES = (0.1, 0.15, 0.25, 0.46, 0.7, 1.0)        # target bits per LR pixel every client is measured at
REFERENCE_BITRATES = (*BITRATES, 1.5, 2.2, 3.3)     # the traditional stream, further up, to match every client
PATH = {"filter": "bicubic", "deblock": False, "encoder": "x264"}   # the host's x264 line
DEFAULT_BPP = degrade.STREAM_BPP
KINDS = ("screen", "photo", "video")
QP_MAX = patches.QP_MAX
# the reference palette of the dataviz method, light mode (validate_palette.js: all checks pass)
SURFACE, INK, SECONDARY, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
ALONE = "#2a78d6"

## 1. Clips

The first validation sources of each kind, 32 frames of 1280×720 from the centre (an image as the window that moved over
it, as notebook 0 showed it), streamed at every bitrate of the traditional stream. The coded size is what counts: x264
holds its bitrate as a ceiling, and a clip that does not need the bits takes fewer.

In [ ]:
manifest = training.load_manifest()
SCALE = manifest["scale"]
video_paths = {f"yt8m-videos/{path.parts[-2]}/{path.stem}": path for path in sources.files("yt8m-videos", "test")}


def clip_of(source):
    """A validation source as HR frames (CLIP_FRAMES, 720, 1280, 3) from its centre, and its frame rate; None if smaller."""
    if source["kind"] == "video":
        frames, fps = sources.load_sequence(video_paths[source["key"]])
    else:
        shown = motion.Motion(**{**source["motion"], "window": tuple(source["motion"]["window"])})
        frames, fps = motion.render(sources.load_image(sources.RAW / source["key"]), shown), IMAGE_FPS
    width, height = SIZE
    if len(frames) < CLIP_FRAMES or frames.shape[1] < height or frames.shape[2] < width:
        return None, fps
    top, left = (frames.shape[1] - height) // 4 * 2, (frames.shape[2] - width) // 4 * 2
    return frames[:CLIP_FRAMES, top:top + height, left:left + width], fps


unique = {}
for source in manifest["splits"]["val"]["sources"]:
    if source["key"] in video_paths or source["kind"] != "video":
        unique.setdefault(source["key"], source)
clips = []   # per source: what it shows, its HR frames and frame rate, and per bitrate its LR frames, coded bits, conditions
for kind in KINDS:
    of_kind = [source for source in unique.values() if source["kind"] == kind]
    taken = 0
    for source in of_kind[::max(1, len(of_kind) // (4 * SOURCES_PER_KIND))]:
        if taken == SOURCES_PER_KIND:
            break
        frames, fps = clip_of(source)
        if frames is None:
            continue
        streams = {}
        for bpp in REFERENCE_BITRATES:
            hr, lr, bits, report = degrade.degrade(frames, degrade.Degradation(scale=SCALE, bpp=bpp, fps=fps, **PATH),
                                                   with_frames=True)
            conditions = np.array([[row["slice_qp"] / QP_MAX, row["deblocked"], row["type"] == "I"] for row in report],
                                  np.float32)
            streams[bpp] = (lr, bits, conditions)
        clips.append({"kind": kind, "key": source["key"], "hr": hr, "fps": fps, "streams": streams})
        taken += 1

print(f"{'':<8}coded bits per LR pixel, at a target of")
print(f"{'':<8}" + "".join(f"{bpp:>7}" for bpp in REFERENCE_BITRATES))
for kind in KINDS:
    of_kind = [clip for clip in clips if clip["kind"] == kind]
    coded = [np.mean([clip["streams"][bpp][1] / clip["streams"][bpp][0][..., 0].size for clip in of_kind])
             for bpp in REFERENCE_BITRATES]
    print(f"{kind:<8}" + "".join(f"{value:7.3f}" for value in coded) + f"   ({len(of_kind)} clips)")

## 2. Clients

Every client's compute is counted from its graph the way the notebooks count it, operations per LR pixel, except
bicubic, which a browser leaves to the GPU's texture sampler and so costs no shader work at all: it is the chart's
baseline, x1 by definition.

In [ ]:
graphs, cost = {}, {"bicubic": 0.0}
for name, file in CLIENTS.items():
    if file is None:
        continue
    path = webexport.MODELS_DIR / file
    model = onnx.load(str(path))
    metadata = webexport.read_metadata(model)
    stored = {init.name for init in model.graph.initializer}
    told = len([value for value in model.graph.input if value.name not in stored]) > 1
    dims = [d.dim_value or None for d in model.graph.input[0].type.tensor_type.shape.dim]
    step = (int(metadata["step_h"]), int(metadata["step"])) if dims[2] else None   # a free size runs the whole frame
    graphs[name] = (webexport.session(path), step, int(metadata["halo"]), told)
    tile = path if step else None
    if tile is None:   # count the work of a told graph on a client tile's shape
        tile = next(webexport.MODELS_DIR.glob({"told 24x2": "upscale_qp_slice_24x2_tile320x180.onnx",
                                               "told fast q24": "upscale_fast_*_tile320x180.onnx"}[name]))
    work = webexport.count_work(tile)
    cost[name] = work["operations"] / work["input_pixels"]


def upscale(name, lr, condition):
    """A client's picture of an LR frame (1, 3, h, w): bicubic, or its graph - a tile graph tile by tile, each tile with
    its halo of real neighbours; a told graph on the whole frame with this frame's condition. The frame's edge is read
    again outward at the border, as the client does."""
    if CLIENTS[name] is None:
        return models.bicubic(lr, SCALE)
    runner, step, halo, told = graphs[name]
    height, width = lr.shape[2:]
    padded = F.pad(lr, (halo,) * 4, mode="replicate").cpu().numpy()
    feeds = lambda x: {runner.get_inputs()[0].name: x, **({runner.get_inputs()[1].name: condition} if told else {})}
    if step is None:
        result = runner.run(None, feeds(padded))[0]
        return torch.from_numpy(result[..., halo * SCALE:-halo * SCALE, halo * SCALE:-halo * SCALE]).to(DEVICE)
    step_h, step_w = step
    assert height % step_h == 0 and width % step_w == 0, "the tiles do not divide the frame"
    out = np.zeros((1, 3, height * SCALE, width * SCALE), np.float32)
    for top in range(0, height, step_h):
        for left in range(0, width, step_w):
            result = runner.run(None, feeds(padded[:, :, top:top + step_h + 2 * halo, left:left + step_w + 2 * halo]))[0]
            out[..., top * SCALE:(top + step_h) * SCALE, left * SCALE:(left + step_w) * SCALE] = \
                result[..., halo * SCALE:(halo + step_h) * SCALE, halo * SCALE:(halo + step_w) * SCALE]
    return torch.from_numpy(out).to(DEVICE)


print(f"{'':<16}{'ops / LR px':>12}")
for name, ops in cost.items():
    print(f"{name:<16}{ops:12,.0f}" + ("   (the GPU's texture sampler)" if name == "bicubic" else ""))

## 3. Measure

Every client on every frame of every clip, at every bitrate it is measured at (bicubic, the traditional stream, at the
higher ones too). Scores are luma SSIM and PSNR over the whole 1280×720 frame, averaged over the clip.

In [ ]:
def to_tensor(images):
    """uint8 (N, h, w, 3) -> float (N, 3, h, w) in [0, 1] on DEVICE."""
    return torch.from_numpy(np.ascontiguousarray(images)).to(DEVICE).permute(0, 3, 1, 2).float() / 255


records = []   # one per clip, bitrate and client
with torch.no_grad():
    for clip in clips:
        frames = len(clip["hr"])
        for bpp, (lr_frames, video_bits, conditions) in clip["streams"].items():
            for name in (list(CLIENTS) if bpp in BITRATES else ["bicubic"]):
                scores = [metrics.per_image(upscale(name, to_tensor(lr_frames[t:t + 1]), conditions[t]),
                                            to_tensor(clip["hr"][t:t + 1])) for t in range(frames)]
                records.append({"kind": clip["kind"], "key": clip["key"], "bpp": bpp, "client": name,
                                "video_bits": video_bits, "lr_pixels": lr_frames[..., 0].size,
                                "ssim": float(torch.cat([s["ssim"] for s in scores]).mean()),
                                "y": float(torch.cat([s["y"] for s in scores]).mean())})
print(f"{len(records):,} scores of {len(clips)} clips")

## 4. The traditional stream

What bits alone buy: bicubic on the stream at every bitrate, by kind. The bits are what H.264
really used, per LR pixel.

In [ ]:
def pick(**match):
    """The records that match, in clip order."""
    return [r for r in records if all(r[key] == value for key, value in match.items())]


for kind in KINDS:
    print(kind)
    print(f"  {'target':<10}" + "".join(f"{bpp:>8}" for bpp in REFERENCE_BITRATES))
    rows = {bpp: pick(kind=kind, bpp=bpp, client="bicubic") for bpp in REFERENCE_BITRATES}
    coded = [np.mean([r["video_bits"] / r["lr_pixels"] for r in rows[bpp]]) for bpp in REFERENCE_BITRATES]
    print(f"  {'coded':<10}" + "".join(f"{value:8.3f}" for value in coded))
    print(f"  {'ssim-y':<10}" + "".join(f"{np.mean([r['ssim'] for r in rows[bpp]]):8.4f}" for bpp in REFERENCE_BITRATES))
    print(f"  {'psnr-y':<10}" + "".join(f"{np.mean([r['y'] for r in rows[bpp]]):8.2f}" for bpp in REFERENCE_BITRATES))

## 5. Worth in bandwidth

Per clip, the traditional stream is a curve of quality against the bits it really took - kept to
its best so far, since a higher bitrate that scores lower is never the one to pick. A client's
score at a bitrate is found on that curve (linear in log bits between the two bitrates around it),
and the bits there, divided by the bits the client's stream used is its worth. Over a kind's clips, the geometric mean; a clip whose score
no bitrate reaches is counted apart (`+n`; `-` when none is reached), and a client scoring under the
traditional stream's lowest bitrate is worth at most what that bitrate gives (`<`). Then quality at the
same bits, which every clip has: the client's score minus the traditional stream's at the bits the
client used, averaged over a kind's clips.

In [ ]:
def plain_curve(record, key):
    """The traditional stream of this record's clip: its bits, and its quality kept to its best so far."""
    plain = sorted((r["video_bits"], r[key]) for r in pick(key=record["key"], client="bicubic"))
    return np.array([b for b, _ in plain], float), np.maximum.accumulate([q for _, q in plain])


def worth(record, key):
    """The traditional stream's bits for this record's score, over the bits the record used: None if no bitrate reaches
    the score, and whether it is only an upper bound (under the lowest bitrate)."""
    bits, curve = plain_curve(record, key)
    quality, used = record[key], record["video_bits"]
    if quality > curve[-1]:
        return None, False
    if quality <= curve[0]:
        return bits[0] / used, True
    i = int(np.searchsorted(curve, quality))   # the first bitrate that reaches it
    share = (quality - curve[i - 1]) / (curve[i] - curve[i - 1])
    return math.exp(math.log(bits[i - 1]) + share * math.log(bits[i] / bits[i - 1])) / used, False


def at_same_bits(record, key):
    """The record's score minus the traditional stream's with the bits the record used (past its highest bitrate,
    its best)."""
    bits, curve = plain_curve(record, key)
    return record[key] - np.interp(math.log(record["video_bits"]), np.log(bits), curve)


def summary(kind, client, bpp, key):
    """The geometric mean of `worth` over a kind's clips, how many no bitrate reaches, and whether any is a bound."""
    found = [worth(r, key) for r in pick(kind=kind, client=client, bpp=bpp)]
    reached = [value for value, _ in found if value is not None]
    mean = math.exp(np.mean(np.log(reached))) if reached else float("nan")
    return mean, len(found) - len(reached), any(bound for value, bound in found if value is not None)


def cell(kind, client, bpp, key):
    mean, unreached, bound = summary(kind, client, bpp, key)
    text = "-" if math.isnan(mean) else ("<" if bound else "") + f"x{mean:.2f}"
    return f"{text + (f' +{unreached}' if unreached else ''):>12}"


def gain(kind, client, bpp, key):
    """`at_same_bits`, averaged over a kind's clips."""
    return float(np.mean([at_same_bits(r, key) for r in pick(kind=kind, client=client, bpp=bpp)]))


upscalers = [name for name in CLIENTS if name != "bicubic"]
for key, metric in (("ssim", "SSIM-Y"), ("y", "PSNR-Y")):
    print(f"worth in bandwidth, for the same {metric}, at a target of")
    print(f"{'':<18}" + "".join(f"{bpp:>12}" for bpp in BITRATES))
    for kind in KINDS:
        print(kind)
        for client in upscalers:
            print(f"  {client:<16}" + "".join(cell(kind, client, bpp, key) for bpp in BITRATES))
    print()

print("quality at the same bits: SSIM-Y / dB luma over the traditional stream with the bits the client's stream used")
print(f"{'':<18}" + "".join(f"{bpp:>16}" for bpp in BITRATES))
for kind in KINDS:
    print(kind)
    for client in upscalers:
        print(f"  {client:<16}" + "".join(f"{gain(kind, client, bpp, 'ssim'):+.4f}/{gain(kind, client, bpp, 'y'):+.2f}".rjust(16)
                                          for bpp in BITRATES))

## 6. Bang for the buck

At the host's default bitrate: what each client costs on the client against what it is worth in bandwidth and what it
adds at the same bits, both for SSIM-Y. The chart is the second, by kind: up is more quality than the traditional
stream with the same traffic, right is more compute on the client.

In [ ]:
print(f"{'':<16}{'client ops':>11}" + "".join(f"{kind + ': worth, same bits':>26}" for kind in KINDS))
for client in upscalers:
    cells = []
    for kind in KINDS:
        mean, unreached, _ = summary(kind, client, DEFAULT_BPP, "ssim")
        text = "-" if math.isnan(mean) else f"x{mean:.2f}"
        cells.append(f"{text}{f' +{unreached}' if unreached else ''}  {gain(kind, client, DEFAULT_BPP, 'ssim'):+.4f}")
    print(f"{client:<16}{cost[client]:11,.0f}" + "".join(f"{c:>26}" for c in cells))
print("\nops per LR pixel; worth: the traditional stream's bits for the same SSIM-Y over the bits used (+n: clips no "
      "bitrate reaches);\nsame bits: SSIM-Y over the traditional stream with the bits used")

fig, axes = plt.subplots(1, len(KINDS), figsize=(13, 5), facecolor=SURFACE)
for ax, kind in zip(axes, KINDS):
    ax.set_facecolor(SURFACE)
    ax.axhline(0.0, color=MUTED, linewidth=1.2, linestyle=(0, (4, 3)), zorder=1)
    for client in upscalers:
        ax.scatter([cost[client]], [gain(kind, client, DEFAULT_BPP, "ssim")], s=70, color=ALONE, edgecolors=SURFACE,
                   linewidths=1.5, zorder=3)
    ax.set_xscale("log")
    names = ax.secondary_xaxis("top")   # the clients, named once along the top, where they sit
    ticks = {}
    for client in upscalers:
        ticks.setdefault(cost[client], []).append(client)
    names.set_xticks(list(ticks), labels=[" / ".join(clients) for clients in ticks.values()])
    names.tick_params(axis="x", labelrotation=40, labelsize=7.5, labelcolor=SECONDARY, color=MUTED, length=3)
    names.minorticks_off()
    for text in names.get_xticklabels():
        text.set_horizontalalignment("left")
    names.spines["top"].set_color(GRID)
    ax.set_title(f"{kind}s" if kind != "video" else "video", color=INK, fontsize=11, loc="left", pad=48)
    ax.set_xlabel("client operations per LR pixel", color=SECONDARY, fontsize=9)
    ax.grid(color=GRID, linewidth=0.8)
    ax.tick_params(colors=SECONDARY, labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(GRID)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:+.3f}"))
axes[0].set_ylabel("SSIM-Y over the traditional stream, same bits", color=SECONDARY, fontsize=9)
fig.text(0.01, 0.02, f"At the host's default bitrate ({DEFAULT_BPP} bits per LR pixel), mean over a kind's clips; each "
         "panel has its own scale. The dashed line is the traditional stream: H.264 + bicubic, which costs the client "
         "nothing.", fontsize=7.5, color=MUTED)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## 7. What it says

*(written after the run)*